# Raw Ridership Cleaning and DiDisc Setup

This notebook is the working record for turning the raw 2025 TCAT ride files into analysis-ready data for the Cornell unlimited-access study. The first legacy blocks are retained as audit notes from exploration; the canonical cleaning decisions now live in `scripts/` and the canonical modeling files live in `data/processed/Analysis_Ready/`.

Recommended workflow: rebuild processed outputs with `scripts/build_analysis_ready_ridership.py`, then use the labeled sections below to inspect cleaning diagnostics, fare taxonomy, direction diagnostics, and treatment/control definitions.

In [ ]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/ and scripts/.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
ANALYSIS_READY_DIR = PROCESSED_DIR / "Analysis_Ready"
CLEANING_DIAG_DIR = PROCESSED_DIR / "_Archive" / "Diagnostics" / "Ridership_Cleaning_Diagnostics"
DIDISC_DIAG_DIR = PROCESSED_DIR / "_Archive" / "Diagnostics" / "DiDisc_Group_Diagnostics"
DIRECTION_DIAG_DIR = PROCESSED_DIR / "_Archive" / "Diagnostics" / "Direction_Diagnostics"
EXPLORATION_DIR = PROCESSED_DIR / "_Archive" / "Exploration" / "Notebook_Exploration"

# Set this to True when you want the notebook to rebuild processed outputs.
# The build reads raw ride files and GTFS directly; no monthly cleaned ride cache is needed.
REBUILD_PROCESSED = False
if REBUILD_PROCESSED:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "build_analysis_ready_ridership.py")],
        check=True,
    )


## Legacy Raw Exploration

The next cells are early exploration of the raw monthly ride files: columns, dtypes, route fields, and farebox text values. They are kept to show how the cleaning questions were discovered, but they are not the current cleaning pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path

In [ ]:
BASE_DIR = RAW_DIR / "TCAT_Rides"

# Keep identifiers and route codes as strings so they do not flip dtypes across monthly files.
STRING_COLUMNS = [
    "bus",
    "route",
    "ttp",
    "text",
    "description",
    "media_text",
    "busmatch",
    "dropfile_parameter",
    "vehicle_vhist",
    "Route_vhist",
    "Trip_vhist",
    "Inbound_Outbound_vhist",
    "Vehicle_Name_vhist",
    "Run_Id_vhist",
    "Run_Name_vhist",
    "Stop_Name_vhist",
    "Operator_Record_Id_vhist",
    "Route_Name_vhist",
    "Stop_Report_vhist",
    "Message_Type_Id_vhist",
    "Stop_Id_vhist",
    "VehicleStatusID_vhist",
    "Veh_Type_Id_vhist",
    "Block_Farebox_Id_vhist",
    "Previous_Stop_Id_vhist",
    "busmatch_vhist",
    "uuid_vhist",
    "event_type_vhist",
    "swiftly_route_id",
    "swiftly_trip_id",
    "swiftly_block_id",
    "route_provenance",
    "corrected_route",
    "service_category",
    "service_day",
    "correction_source",
]

NUMERIC_DTYPES = {
    "tr_seq": "Float64",
    "amt": "Float64",
    "lat_vhist": "Float64",
    "long_vhist": "Float64",
    "speed_vhist": "Float64",
    "direction_vhist": "Float64",
    "actualHeadway_vhist": "Float64",
    "Deviation_vhist": "Float64",
    "Scheduled_Headway_vhist": "Float64",
    "Target_Headway_vhist": "Float64",
    "Confidence_Level_vhist": "Float64",
    "Stop_Dwell_Time_vhist": "Float64",
    "StationaryDuration_vhist": "Float64",
    "OdometerValue_vhist": "Float64",
    "Distance_vhist": "Float64",
    "canonical_rider": "Int64",
    "corrected_rider": "Int64",
}

BOOLEAN_COLUMNS = [
    "is_cornell",
    "has_fare",
    "has_apc",
    "has_swiftly",
    "is_revenue_rider",
]

DATETIME_COLUMNS = [
    "ts",
    "time_vhist",
    "Server_Time_vhist",
    "Departure_Time_vhist",
    "ts_vhist_vhist",
    "ts_vhist",
    "ts_event",
]

RAW_RIDE_DTYPES = {column: "string" for column in STRING_COLUMNS}
RAW_RIDE_DTYPES.update(NUMERIC_DTYPES)


def read_ride_month(file_path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        file_path,
        dtype=RAW_RIDE_DTYPES,
        parse_dates=DATETIME_COLUMNS,
        true_values=["true", "True"],
        false_values=["false", "False"],
        na_values=["NULL"],
        keep_default_na=True,
        low_memory=False,
    )
    df[BOOLEAN_COLUMNS] = df[BOOLEAN_COLUMNS].astype("boolean")
    df["service_day"] = pd.to_datetime(df["service_day"], format="%y%m%d", errors="coerce")
    return df

In [ ]:
months = []
for month in range(1, 13):
    month_str = f"{month:02d}"
    file_path = BASE_DIR / f"canonical_riders_2025_{month_str}.csv"
    df_month = read_ride_month(file_path)
    months.append(df_month)

tcat_2025_rides = pd.concat(months, ignore_index=True)

In [ ]:
tcat_2025_rides.columns

In [ ]:
print(tcat_2025_rides.dtypes)

In [ ]:
tcat_2025_rides.shape

In [ ]:
print(tcat_2025_rides.head(5))

In [ ]:
tcat_2025_rides['direction_vhist'].unique()

In [ ]:
print(tcat_2025_rides['text'].unique())

In [ ]:
print(tcat_2025_rides['route'].unique())

In [ ]:
print(tcat_2025_rides['corrected_route'].unique())

In [ ]:
tcat_2025_rides['corrected_route'].unique()

In [ ]:
print(tcat_2025_rides['Route_vhist'].unique())

In [ ]:
print(tcat_2025_rides['Route_Name_vhist'].unique())

In [ ]:
tcat_2025_rides['swiftly_route_id'].unique()

In [ ]:
tcat_2025_rides[['route', 'corrected_route', 'Route_vhist', 'Route_Name_vhist']].isna().sum()

## Legacy Route-Resolution Exploration

These cells investigate `route`, `corrected_route`, `Route_vhist`, through-running route codes, and GTFS stop/route mismatches. The repeatable version of this work is now `scripts/build_analysis_ready_ridership.py`, which resolves routes in memory and writes route-resolution summaries in `Analysis_Ready/`.

In [ ]:
# Remove label text and normalize the vehicle-history route code for comparison.
ROUTE_CODE_NORMALIZATION = {
    "145": "14S",
}

tcat_2025_rides["Route_vhist_normalized"] = tcat_2025_rides["Route_vhist"].replace(
    ROUTE_CODE_NORMALIZATION
)

tcat_2025_rides["Route_Name_vhist_clean"] = (
    tcat_2025_rides["Route_Name_vhist"]
    .str.replace(r"^\s*Route\s*", "", regex=True)
    .str.strip()
)

route_vhist_both_missing = (
    tcat_2025_rides["Route_vhist_normalized"].isna()
    & tcat_2025_rides["Route_Name_vhist_clean"].isna()
)
route_vhist_both_present = (
    tcat_2025_rides["Route_vhist_normalized"].notna()
    & tcat_2025_rides["Route_Name_vhist_clean"].notna()
)
route_vhist_values_match = tcat_2025_rides["Route_vhist_normalized"].eq(
    tcat_2025_rides["Route_Name_vhist_clean"]
)

tcat_2025_rides["route_name_matches_route_vhist"] = (
    route_vhist_both_missing | (route_vhist_both_present & route_vhist_values_match)
).astype("boolean")

In [ ]:
route_vhist_check = pd.Series(
    {
        "total_rows": len(tcat_2025_rides),
        "both_missing_rows": route_vhist_both_missing.sum(),
        "comparable_rows": route_vhist_both_present.sum(),
        "matching_comparable_rows": (route_vhist_both_present & route_vhist_values_match).sum(),
        "mismatching_comparable_rows": (
            route_vhist_both_present & ~route_vhist_values_match.fillna(False)
        ).sum(),
        "partial_missing_rows": (
            tcat_2025_rides["Route_vhist_normalized"].isna()
            ^ tcat_2025_rides["Route_Name_vhist_clean"].isna()
        ).sum(),
    },
    name="rows",
)

route_vhist_mismatch_summary = (
    tcat_2025_rides.loc[
        route_vhist_both_present & ~route_vhist_values_match.fillna(False),
        [
            "Route_vhist",
            "Route_vhist_normalized",
            "Route_Name_vhist",
            "Route_Name_vhist_clean",
        ],
    ]
    .value_counts(dropna=False)
    .rename("rows")
    .reset_index()
)

display(route_vhist_check)
display(route_vhist_mismatch_summary)

In [ ]:
tcat_2025_rides["corrected_route_normalized"] = tcat_2025_rides[
    "corrected_route"
].replace(ROUTE_CODE_NORMALIZATION)

corrected_route_both_missing = (
    tcat_2025_rides["corrected_route_normalized"].isna()
    & tcat_2025_rides["Route_vhist_normalized"].isna()
)
corrected_route_both_present = (
    tcat_2025_rides["corrected_route_normalized"].notna()
    & tcat_2025_rides["Route_vhist_normalized"].notna()
)
corrected_route_raw_values_match = tcat_2025_rides["corrected_route"].eq(
    tcat_2025_rides["Route_vhist_normalized"]
)
corrected_route_normalized_values_match = tcat_2025_rides[
    "corrected_route_normalized"
].eq(tcat_2025_rides["Route_vhist_normalized"])

tcat_2025_rides["corrected_route_matches_route_vhist_normalized"] = (
    corrected_route_both_missing
    | (corrected_route_both_present & corrected_route_normalized_values_match)
).astype("boolean")

In [ ]:
corrected_route_check = pd.Series(
    {
        "total_rows": len(tcat_2025_rides),
        "both_missing_rows": corrected_route_both_missing.sum(),
        "comparable_rows": corrected_route_both_present.sum(),
        "matching_comparable_rows_raw_corrected_route": (
            corrected_route_both_present & corrected_route_raw_values_match
        ).sum(),
        "mismatching_comparable_rows_raw_corrected_route": (
            corrected_route_both_present
            & ~corrected_route_raw_values_match.fillna(False)
        ).sum(),
        "matching_comparable_rows_normalized_corrected_route": (
            corrected_route_both_present & corrected_route_normalized_values_match
        ).sum(),
        "mismatching_comparable_rows_normalized_corrected_route": (
            corrected_route_both_present
            & ~corrected_route_normalized_values_match.fillna(False)
        ).sum(),
        "partial_missing_rows": (
            tcat_2025_rides["corrected_route_normalized"].isna()
            ^ tcat_2025_rides["Route_vhist_normalized"].isna()
        ).sum(),
    },
    name="rows",
)

corrected_route_raw_mismatch_summary = (
    tcat_2025_rides.loc[
        corrected_route_both_present & ~corrected_route_raw_values_match.fillna(False),
        ["corrected_route", "Route_vhist", "Route_vhist_normalized"],
    ]
    .value_counts(dropna=False)
    .rename("rows")
    .reset_index()
)

corrected_route_normalized_mismatch_summary = (
    tcat_2025_rides.loc[
        corrected_route_both_present
        & ~corrected_route_normalized_values_match.fillna(False),
        [
            "corrected_route",
            "corrected_route_normalized",
            "Route_vhist",
            "Route_vhist_normalized",
        ],
    ]
    .value_counts(dropna=False)
    .rename("rows")
    .reset_index()
)

corrected_route_missing_vhist_summary = (
    tcat_2025_rides.loc[
        tcat_2025_rides["corrected_route_normalized"].notna()
        & tcat_2025_rides["Route_vhist_normalized"].isna(),
        "corrected_route_normalized",
    ]
    .value_counts(dropna=False)
    .rename("rows")
    .reset_index(name="rows")
)

display(corrected_route_check)
display(corrected_route_raw_mismatch_summary)
display(corrected_route_normalized_mismatch_summary)
display(corrected_route_missing_vhist_summary)

In [ ]:
OUTPUT_DIR = EXPLORATION_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

corrected_route_missing_vhist_summary_path = (
    OUTPUT_DIR / "corrected_route_missing_vhist_summary.csv"
)
corrected_route_missing_vhist_summary.to_csv(
    corrected_route_missing_vhist_summary_path,
    index=False,
)

corrected_route_missing_vhist_summary_path

In [ ]:
from functools import lru_cache
import zipfile

gtfs_zips = sorted((RAW_DIR / "TCAT_GTFS").glob("*.zip"))
if not gtfs_zips:
    raise FileNotFoundError(f"No GTFS zip files found in {RAW_DIR / 'TCAT_GTFS'}")

GTFS_ROUTES_PATH = gtfs_zips[-1]
with zipfile.ZipFile(GTFS_ROUTES_PATH) as gtfs_zip:
    with gtfs_zip.open("routes.txt") as routes_file:
        gtfs_routes = pd.read_csv(routes_file, dtype="string")

route_id_to_short = dict(
    zip(gtfs_routes["route_id"].astype(str), gtfs_routes["route_short_name"].astype(str))
)

# Keep route 22 and any other vehicle-history routes present in the data even if
# they are absent from the attached GTFS routes file.
for route_code in tcat_2025_rides["Route_vhist_normalized"].dropna().astype(str).unique():
    route_id_to_short.setdefault(route_code, route_code)

canonical_route_shorts_for_diagnostic = set(route_id_to_short.values())
route_id_tokens_for_diagnostic = sorted(route_id_to_short, key=lambda code: (-len(code), code))


@lru_cache(None)
def split_route_code_into_known_routes(route_code: str) -> tuple[tuple[str, ...], ...]:
    if not route_code:
        return ((),)

    splits = []
    for token in route_id_tokens_for_diagnostic:
        if route_code.startswith(token):
            for remaining_split in split_route_code_into_known_routes(route_code[len(token) :]):
                splits.append((route_id_to_short[token],) + remaining_split)
    return tuple(splits)


def format_through_run_splits(route_code: str) -> str:
    route_code = str(route_code)
    splits = [split for split in split_route_code_into_known_routes(route_code) if len(split) >= 2]
    unique_splits = list(dict.fromkeys(splits))
    return " | ".join(" + ".join(split) for split in unique_splits)


corrected_route_summary = (
    tcat_2025_rides["corrected_route"]
    .dropna()
    .astype(str)
    .value_counts()
    .rename_axis("corrected_route")
    .reset_index(name="rows")
)
corrected_route_summary["corrected_route_normalized"] = corrected_route_summary[
    "corrected_route"
].replace(ROUTE_CODE_NORMALIZATION)
corrected_route_summary["is_canonical_route"] = corrected_route_summary[
    "corrected_route_normalized"
].isin(canonical_route_shorts_for_diagnostic)
corrected_route_summary["through_run_parse"] = corrected_route_summary[
    "corrected_route"
].map(format_through_run_splits)

corrected_route_through_run_candidates = corrected_route_summary.loc[
    corrected_route_summary["through_run_parse"].ne("")
].copy()
corrected_route_noncanonical_summary = corrected_route_summary.loc[
    ~corrected_route_summary["is_canonical_route"]
].copy()

display(corrected_route_through_run_candidates)
display(corrected_route_noncanonical_summary.head(40))


In [ ]:
through_run_route_codes = set(corrected_route_through_run_candidates["corrected_route"])
corrected_route_through_run_rows = tcat_2025_rides.loc[
    tcat_2025_rides["corrected_route"].astype(str).isin(through_run_route_codes),
    [
        "ts",
        "service_day",
        "bus",
        "tr_seq",
        "route",
        "corrected_route",
        "route_provenance",
        "correction_source",
        "service_category",
        "has_swiftly",
        "Route_vhist",
        "Route_vhist_normalized",
        "Route_Name_vhist",
        "swiftly_route_id",
        "swiftly_trip_id",
    ],
].copy()
corrected_route_through_run_rows["through_run_parse"] = corrected_route_through_run_rows[
    "corrected_route"
].astype(str).map(format_through_run_splits)

corrected_route_through_run_bus_day_summary = (
    corrected_route_through_run_rows.groupby(
        ["corrected_route", "through_run_parse", "service_day", "bus"],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["rows", "corrected_route"], ascending=[False, True])
)

corrected_route_through_run_candidates_path = (
    OUTPUT_DIR / "corrected_route_through_run_candidates.csv"
)
corrected_route_noncanonical_summary_path = (
    OUTPUT_DIR / "corrected_route_noncanonical_summary.csv"
)
corrected_route_through_run_rows_path = OUTPUT_DIR / "corrected_route_through_run_rows.csv"
corrected_route_through_run_bus_day_summary_path = (
    OUTPUT_DIR / "corrected_route_through_run_bus_day_summary.csv"
)

corrected_route_through_run_candidates.to_csv(
    corrected_route_through_run_candidates_path, index=False
)
corrected_route_noncanonical_summary.to_csv(
    corrected_route_noncanonical_summary_path, index=False
)
corrected_route_through_run_rows.to_csv(corrected_route_through_run_rows_path, index=False)
corrected_route_through_run_bus_day_summary.to_csv(
    corrected_route_through_run_bus_day_summary_path, index=False
)

[
    corrected_route_through_run_candidates_path,
    corrected_route_noncanonical_summary_path,
    corrected_route_through_run_rows_path,
    corrected_route_through_run_bus_day_summary_path,
]

In [ ]:
# Current route-resolution summary from the unified raw-to-analysis build.
route_resolution_summary = pd.read_csv(ANALYSIS_READY_DIR / "route_resolution_summary.csv")
route_resolution_by_month = pd.read_csv(ANALYSIS_READY_DIR / "route_resolution_by_month.csv")

display(route_resolution_summary)
route_resolution_by_month.pivot_table(
    index="source_month",
    columns="normalized_route_source",
    values="rows",
    aggfunc="sum",
    fill_value=0,
).reset_index()

In [ ]:
# The old monthly route-cleaning cache is no longer required.
# Use these compact summaries instead of reading archived intermediate CSVs.
resolved_rows = route_resolution_summary.loc[
    ~route_resolution_summary["normalized_route_source"].str.startswith("unresolved")
    & route_resolution_summary["normalized_route_source"].ne("excluded_text_removed"),
    "rows",
].sum()
unresolved_rows = route_resolution_summary.loc[
    route_resolution_summary["normalized_route_source"].str.startswith("unresolved"),
    "rows",
].sum()
excluded_text_rows = route_resolution_summary.loc[
    route_resolution_summary["normalized_route_source"].eq("excluded_text_removed"),
    "rows",
].sum()

pd.Series(
    {
        "resolved_rows": resolved_rows,
        "unresolved_route_rows": unresolved_rows,
        "excluded_text_rows": excluded_text_rows,
    },
    name="rows",
)

## Route-Column Agreement Checks

The current build resolves routes conservatively: `Route_vhist` is primary; when it is missing, `corrected_route` is kept only if the date-appropriate GTFS feed confirms it is a real route. Through-running strings, GTFS stop IDs in `corrected_route`, non-GTFS values, and missing route values are dropped rather than inferred.

These checks compare the route columns only when both values are canonical GTFS route codes.

In [ ]:
route_resolution_summary = pd.read_csv(ANALYSIS_READY_DIR / "route_resolution_summary.csv")
route_column_coverage = pd.read_csv(ANALYSIS_READY_DIR / "route_column_coverage.csv")
route_column_pairwise = pd.read_csv(ANALYSIS_READY_DIR / "route_column_pairwise_disagreements.csv")
route_column_top_disagreements = pd.read_csv(ANALYSIS_READY_DIR / "route_column_top_disagreements.csv")

display(route_resolution_summary)
display(route_column_coverage)
route_column_pairwise

In [ ]:
# Largest disagreements. These are concentrated in the raw `route` field.
route_column_top_disagreements.head(25)

## Legacy Fare, Media, and Dropfile Exploration

These cells inspect fare descriptions, media text, `dropfile_parameter`, and Cornell flags. The current fare recode is centralized in `scripts/build_analysis_ready_ridership.py`; missing/invalid record decisions are summarized later from `_Archive/Diagnostics/Ridership_Cleaning_Diagnostics/` and `Analysis_Ready/`.

In [ ]:
tcat_2025_rides['description'].unique()

In [ ]:
tcat_2025_rides['media_text'].unique()

In [ ]:
tcat_2025_rides['dropfile_parameter'].unique()

In [ ]:
tcat_2025_rides['is_cornell'].unique()

In [ ]:
tcat_2025_rides[['description', 'media_text', 'dropfile_parameter', 'is_cornell']].isna().sum()

In [ ]:
dropfile_check = tcat_2025_rides[
    ["dropfile_parameter", "text", "description", "media_text", "is_cornell"]
].copy()
dropfile_check["dropfile_parameter_clean"] = (
    dropfile_check["dropfile_parameter"].fillna("").astype(str).str.strip()
)

dropfile_nonblank = dropfile_check["dropfile_parameter_clean"].ne("")
dropfile_nonblank_nonexception = dropfile_nonblank & dropfile_check[
    "dropfile_parameter_clean"
].ne("EXCEPTION")

dropfile_expected_match = (
    dropfile_check["text"].eq("Special card")
    & dropfile_check["description"].eq("Cornell Card")
    & dropfile_check["media_text"].eq("Cornell")
    & dropfile_check["is_cornell"].eq(True)
)

dropfile_condition_mismatch_mask = dropfile_nonblank_nonexception & ~dropfile_expected_match
dropfile_condition_summary = pd.Series(
    {
        "total_rows": len(dropfile_check),
        "dropfile_blank_rows": (~dropfile_nonblank).sum(),
        "dropfile_nonblank_rows": dropfile_nonblank.sum(),
        "dropfile_exception_rows": dropfile_check["dropfile_parameter_clean"].eq("EXCEPTION").sum(),
        "dropfile_nonblank_nonexception_rows": dropfile_nonblank_nonexception.sum(),
        "condition_match_rows": (dropfile_nonblank_nonexception & dropfile_expected_match).sum(),
        "condition_mismatch_rows": dropfile_condition_mismatch_mask.sum(),
    },
    name="rows",
)

dropfile_combo_summary = (
    dropfile_check.loc[dropfile_nonblank]
    .groupby(
        ["dropfile_parameter_clean", "text", "description", "media_text", "is_cornell"],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)

dropfile_blank_combo_summary = (
    dropfile_check.loc[~dropfile_nonblank]
    .groupby(["text", "description", "media_text", "is_cornell"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)

dropfile_condition_mismatch_rows = tcat_2025_rides.loc[
    dropfile_condition_mismatch_mask,
    [
        "ts",
        "bus",
        "route",
        "corrected_route",
        "dropfile_parameter",
        "text",
        "description",
        "media_text",
        "is_cornell",
        "service_day",
    ],
].copy()

display(dropfile_condition_summary)
display(dropfile_combo_summary)
display(dropfile_blank_combo_summary.head(30))
display(dropfile_condition_mismatch_rows)

In [ ]:
dropfile_condition_summary.to_csv(
    EXPLORATION_DIR / "dropfile_parameter_condition_summary.csv", header=True
)
dropfile_combo_summary.to_csv(
    EXPLORATION_DIR / "dropfile_parameter_combo_summary.csv", index=False
)
dropfile_blank_combo_summary.to_csv(
    EXPLORATION_DIR / "dropfile_parameter_blank_combo_summary.csv", index=False
)
dropfile_condition_mismatch_rows.to_csv(
    EXPLORATION_DIR / "dropfile_parameter_condition_mismatch_rows.csv", index=False
)

[
    EXPLORATION_DIR / "dropfile_parameter_condition_summary.csv",
    EXPLORATION_DIR / "dropfile_parameter_combo_summary.csv",
    EXPLORATION_DIR / "dropfile_parameter_blank_combo_summary.csv",
    EXPLORATION_DIR / "dropfile_parameter_condition_mismatch_rows.csv",
]

In [ ]:
EXPECTED_MEDIA_TEXT_BY_DESCRIPTION = {
    "Cornell Card": "Cornell",
    "Second Left Arrow 16": "Cornell",
    "Ithaca College Pass": "IC Pass",
    "Third Left Arrow 17": "IC",
    "Fourth Left Arrow 18": "TC3",
    "15 Ride Pass Mobile": "15 RIDE",
    "15 RIDE TCARD": "15 RIDE",
    "15 RIDE HALF-FARE TCARD": "15 RIDE",
    "15 Ride Pass Half Fare Mobile": "15 RIDE",
    "1 Day Pass Mobile": "1DAYPASS",
    "Pay As You Go": "PayAsUGo",
    "PayAsYouGo": "PayAsUGo",
    "Monthly Pass Mobile": "MONTHLY",
    "31-DAY TCARD": "31 DAY",
    "RA1 Adult 18-59": "TCARD",
    "RA2  Senior 60 & Up": "HalfFare",
    "LA1 Youth": "Youth",
    "RA4  Transfer": "Transfer",
    "1 Ride Pass Mobile": "1 RIDE",
    "1 Ride Half Mobile": "1 RIDE",
    "2 Ride Pass Mobile": "2 RIDE",
    "2 Ride Half Mobile": "2 RIDE",
    "1-DAY TCARD": "1 DAY",
    "RA3  Disabled": "CityPass",
    "Weekly Pass Mobile": "7DAYPASS",
    "2 Day Pass Mobile": "2DAYPASS",
    "5 Day Pass Mobile": "5DAYPASS",
    "ANNUAL PASS": "ANNUAL",
    "5-DAY TCARD": "5 DAY",
    "2-DAY TCARD": "2 DAY",
    "Empty TTP": "",
}

description_media_check = tcat_2025_rides[["description", "media_text"]].copy()
description_media_check["description_clean"] = (
    description_media_check["description"].fillna("").astype(str).str.strip()
)
description_media_check["media_text_clean"] = (
    description_media_check["media_text"].fillna("").astype(str).str.strip()
)
description_media_check["expected_media_text"] = description_media_check[
    "description_clean"
].map(EXPECTED_MEDIA_TEXT_BY_DESCRIPTION)

description_media_both_blank = description_media_check["description_clean"].eq("") & description_media_check[
    "media_text_clean"
].eq("")
description_media_matched = description_media_check["media_text_clean"].eq(
    description_media_check["expected_media_text"]
)

description_media_check["match_status"] = np.select(
    [
        description_media_both_blank,
        description_media_matched,
        description_media_check["expected_media_text"].isna(),
    ],
    ["both_blank", "matched", "unmapped_description"],
    default="mismatch",
)

description_media_pair_summary = (
    description_media_check.groupby(
        ["description_clean", "media_text_clean", "expected_media_text", "match_status"],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)

description_media_match_status_summary = (
    description_media_check["match_status"]
    .value_counts(dropna=False)
    .rename_axis("match_status")
    .reset_index(name="rows")
)

description_media_mismatch_rows = tcat_2025_rides.loc[
    description_media_check["match_status"].isin(["mismatch", "unmapped_description"]),
    [
        "ts",
        "bus",
        "route",
        "corrected_route",
        "text",
        "description",
        "media_text",
        "ttp",
        "amt",
        "service_category",
        "service_day",
    ],
].copy()
description_media_mismatch_rows["expected_media_text"] = description_media_check.loc[
    description_media_mismatch_rows.index, "expected_media_text"
]
description_media_mismatch_rows["match_status"] = description_media_check.loc[
    description_media_mismatch_rows.index, "match_status"
]

display(description_media_match_status_summary)
display(description_media_pair_summary)
display(description_media_mismatch_rows)

In [ ]:
description_media_pair_summary.to_csv(
    EXPLORATION_DIR / "description_media_pair_summary.csv", index=False
)
description_media_match_status_summary.to_csv(
    EXPLORATION_DIR / "description_media_match_status_summary.csv", index=False
)
description_media_mismatch_rows.to_csv(
    EXPLORATION_DIR / "description_media_mismatch_rows.csv", index=False
)

[
    EXPLORATION_DIR / "description_media_pair_summary.csv",
    EXPLORATION_DIR / "description_media_match_status_summary.csv",
    EXPLORATION_DIR / "description_media_mismatch_rows.csv",
]

## Current Script-Generated Analysis Sections

From this point down, the notebook reads repeatable outputs generated by the scripts. These are the sections to rely on for the study writeup and modeling decisions.

## Difference-in-discontinuities treatment/control diagnostics

This section uses the cleaned raw ride files to diagnose which route and stop cells are plausibly exposed to the Cornell 6 pm/weekend free-ride discontinuity. The key design distinction is:

- `Second Left Arrow 16` is a post-cutoff Cornell override/compliance signal, not a stable pre/post rider group.
- `Cornell Card` riders already have unlimited access, so they are a Cornell-exposure proxy and a useful placebo/negative-control fare group, but not the marginal treated group for the 6 pm price change.
- The main treated group should be Cornell-exposed route/stop cells, using all observed boardings as the outcome, because Cornell students who paid regular fare before 6 pm can appear as cash, mobile, TCard, transfer, annual pass, etc.


In [ ]:
from pathlib import Path
import pandas as pd

DIAG_DIR = DIDISC_DIAG_DIR

# Rebuild these files with:
# /Users/stevenzhou/miniconda3/bin/conda run -n spatial python scripts/_archive_diagnostics/didisc_group_diagnostics.py

route_near_6 = pd.read_csv(DIAG_DIR / "route_near_6pm_summary.csv")
route_sets = pd.read_csv(DIAG_DIR / "candidate_route_set_diagnostics.csv")
stop_exposure = pd.read_csv(DIAG_DIR / "stop_cornell_exposure_summary.csv")
fare_period = pd.read_csv(DIAG_DIR / "fare_group_by_policy_period.csv", index_col=0)
hourly_fares = pd.read_csv(DIAG_DIR / "old_route_sets_hourly_fare_groups.csv")


In [ ]:
# Fare-media behavior confirms that the override is concentrated in free-policy periods.
fare_period

In [ ]:
# Candidate route sets. The old no-Cornell set is not clean in the raw fare data.
route_sets

In [ ]:
# Route evidence around the weekday 6 pm cutoff.
# Keep routes with enough service on both sides of the cutoff before assigning treated/control status.
cols = [
    "route",
    "boards_pre",
    "cornell_card_share_pre",
    "cornell_override_share_pre",
    "boards_post",
    "cornell_card_share_post",
    "cornell_override_share_post",
    "boards_sum",
]
route_near_6.loc[route_near_6["boards_sum"] >= 1_000, cols].head(30)

In [ ]:
# Stop-level Cornell exposure based only on pre-6 weekday Cornell Card share.
# These are good candidates for treated campus stop cells when they also have service after 6 pm.
stop_cols = [
    "Stop_Id_vhist",
    "Stop_Name_vhist",
    "boards",
    "routes",
    "cornell_card_share",
    "cornell_any_observed_share",
]
stop_exposure.loc[stop_exposure["boards"] >= 500, stop_cols].head(30)

### Working definition for the DiDisc design

Recommended primary definition:

- Treated/exposed cells: route-stop or route-day cells with high Cornell exposure, measured before the cutoff using `Cornell Card` share or known campus stops/routes. For a clean binary version, start with routes `30`, `81`, `32`, `82`, and `51`; add `37`, `31`, `52`, and similar routes only in sensitivity checks if they have enough service on both sides of 6 pm.
- Control cells: low-Cornell-exposure cells with service on both sides of 6 pm. The cleanest route controls in the raw data are `11` and `14`; `13` and `14S` are possible robustness controls but are weaker because `13` has a large override share and `14S` has little post-6 service.
- Better than a hard binary: use continuous Cornell exposure, e.g. pre-6 weekday `Cornell Card` share by route-stop or route, interacted with the post-6 discontinuity.

Avoid as the main design:

- Do not call `Second Left Arrow 16` the treated group. It is partly a result of treatment and can include driver override cases.
- Do not use `Cornell Card` as the main control group. Those riders are already unlimited-access riders before and after 6 pm. Use them as an exposure proxy or placebo check instead.
- Do not use the old `no_cornell` route list without revision. In the raw fare data, routes `17`, `21`, `36`, `65`, and `67` show substantial Cornell Card shares before 6 pm.


### Outcome coding note

For the main ridership DiDisc, use total boardings in the exposed route/stop cell as the outcome, or use total boardings excluding `Cornell Card` as a sensitivity check for non-already-unlimited riders. Keep `Second Left Arrow 16` in the outcome because that is how many post-6 Cornell-eligible rides are recorded.

Use `Second Left Arrow 16` itself only as a secondary fare-media/compliance outcome, not as the main treated group. A jump in that field after 6 pm is partly mechanical because the farebox records the policy interaction.


## Analysis-ready cleaning layer

The remaining cleaning issues that matter for the DiDisc study are mostly analytical, not raw parsing:

- Use `event_time = ts_event -> ts -> ts_vhist`; raw `ts` is missing for APC-only rows, but `event_time` is complete.
- Treat `Stop_Id_vhist == 0` / `Undefined` as missing stop information.
- Canonicalize stop names by `Stop_Id_vhist`, because the same stop appears under several truncated names.
- Flag and drop duplicate-looking fare events from analysis panels.
- Keep `Second Left Arrow 16` in total boardings, but flag off-policy overrides separately.
- Use cleaned route and route-stop exposure labels for treated/control definitions.
- Recode fare descriptions into a two-level taxonomy: `Free`, `Half Fare`, and `Regular Fares`, with detailed categories for Cornell, IC, TC3, transfers, ride-based half fares, ride-based regular fares, and period-based regular fares.
- Do not drop `amt > 10` rows by default; those are mostly pass products, not impossible boardings.
- Drop `Empty TTP` from analysis panels, while keeping and flagging `EXCEPTION`, blank `Got fare`, and blank APC-only rows.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

ANALYSIS_READY_DIR = PROCESSED_DIR / "Analysis_Ready"
CLEANING_DIAG_DIR = PROCESSED_DIR / "_Archive" / "Diagnostics" / "Ridership_Cleaning_Diagnostics"

# Set to True after changing scripts or processed inputs.
REBUILD_ANALYSIS_READY = False
if REBUILD_ANALYSIS_READY:
    subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "build_analysis_ready_ridership.py")], check=True)


In [ ]:
cleaning_summary = pd.read_csv(
    ANALYSIS_READY_DIR / "analysis_ready_cleaning_summary.csv", index_col=0
).rename(columns={"value": "rows"})

quality_summary = pd.read_csv(
    CLEANING_DIAG_DIR / "quality_summary.csv", index_col=0
).rename(columns={"value": "rows"})

event_time_source = pd.read_csv(CLEANING_DIAG_DIR / "event_time_source_summary.csv")
duplicate_summary = pd.read_csv(CLEANING_DIAG_DIR / "duplicate_candidate_summary.csv")
route_groups = pd.read_csv(ANALYSIS_READY_DIR / "route_group_lookup.csv")
route_stop_groups = pd.read_csv(ANALYSIS_READY_DIR / "route_stop_group_lookup.csv")
stop_name_lookup = pd.read_csv(ANALYSIS_READY_DIR / "stop_name_lookup.csv")

display(cleaning_summary)
display(quality_summary)
display(event_time_source)
display(duplicate_summary)


In [ ]:
# Fare category taxonomy after recoding descriptions.
fare_category_lookup = pd.read_csv(ANALYSIS_READY_DIR / "fare_category_lookup.csv")
fare_category_summary = pd.read_csv(
    ANALYSIS_READY_DIR / "fare_category_description_summary.csv"
)
fare_family_policy = pd.read_csv(
    CLEANING_DIAG_DIR / "fare_family_policy_corrected_riders.csv",
    index_col=0,
)
fare_category_policy = pd.read_csv(
    CLEANING_DIAG_DIR / "fare_category_policy_corrected_riders.csv",
    index_col=0,
)

display(fare_category_lookup)
display(fare_family_policy)
display(fare_category_policy)
display(fare_category_summary)

In [ ]:
# Missing/invalid record diagnostics.
record_issue_summary = pd.read_csv(
    CLEANING_DIAG_DIR / "record_issue_summary.csv"
)
record_issue_by_route = pd.read_csv(
    CLEANING_DIAG_DIR / "record_issue_by_route.csv"
)
analysis_record_issue_summary = pd.read_csv(
    ANALYSIS_READY_DIR / "analysis_record_issue_summary.csv"
)
upstream_excluded_text = pd.read_csv(
    CLEANING_DIAG_DIR / "upstream_excluded_text_summary.csv"
)

display(upstream_excluded_text)
display(record_issue_summary)
display(analysis_record_issue_summary)
record_issue_by_route.sort_values(["issue", "rows"], ascending=[True, False]).head(40)

Cleaning choices for missing/invalid records:

- `Auto logoff (no activity)` and `Direction change` are already removed by the route-cleaning script before the analysis-ready layer.
- `Empty TTP` is treated as an invalid/empty fare product and dropped from analysis panels.
- `dropfile_parameter == EXCEPTION` is kept as a valid Cornell Card ride but flagged, because it consistently matches `Special card` / `Cornell Card` / `Cornell`.
- Blank-description `Got fare` records are treated as ride-based regular fares and flagged as `fare_record_missing_description`.
- Blank APC-only rows are kept for total ridership and flagged as `apc_only_missing_fare_description`; they should not be used for fare-mix analysis.


The fare recode now uses these analysis categories:

- `Cornell`: `Cornell Card`
- `Cornell Override`: `Second Left Arrow 16`
- `IC`: `Ithaca College Pass`
- `IC Override`: `Third Left Arrow 17`
- `Free`: `Free Fares`, `Transfer`, `Cornell`, `Cornell Override`, `IC`, `IC Override`, and `TC3 Override`
- `Half Fare`: `Ride-based Half Fares`
- `Regular Fares`: `Ride-based Regular Fares` and `Period-based Regular Fares`
- `Unspecified`: blank APC-only fare descriptions; `Empty TTP` is mapped for diagnostics but dropped from analysis panels


In [ ]:
# Route-level treatment/control labels after cleaning.
route_groups

In [ ]:
# Route-stop treatment/control cells based on pre-6 weekday Cornell Card share.
route_stop_group_counts = (
    route_stop_groups["route_stop_group"]
    .value_counts()
    .rename_axis("route_stop_group")
    .rename("cells")
    .reset_index()
)
display(route_stop_group_counts)

route_stop_groups.query(
    "route_stop_group in ['treated_high_cornell_route_stop', 'control_low_cornell_route_stop']"
).head(30)

In [ ]:
# Stop-name variants that are now canonicalized by stop ID.
stop_name_lookup.query("stop_name_variant_count > 1").head(25)

In [ ]:
# Route-level 15-minute panel ready for DiDisc models.
route_panel = pd.read_parquet(ANALYSIS_READY_DIR / "route_15min_panel.parquet")

didisc_route_panel = route_panel.query(
    "weekday_near_6pm and route_group_primary in "
    "['treated_high_cornell_route', 'control_low_cornell_route']"
).copy()
didisc_route_panel["treated"] = didisc_route_panel["route_group_primary"].eq(
    "treated_high_cornell_route"
).astype(int)
didisc_route_panel["post"] = didisc_route_panel["post_6pm"].astype(int)
didisc_route_panel["running_minutes"] = didisc_route_panel["minutes_since_18"]

didisc_route_panel.head()

In [ ]:
# Route-stop 15-minute panel for the cleaner exposure/control design.
# This file is larger than the route panel; load it only when modeling route-stop cells.
route_stop_panel = pd.read_parquet(ANALYSIS_READY_DIR / "route_stop_15min_panel.parquet")

didisc_route_stop_panel = route_stop_panel.query(
    "weekday_near_6pm and route_stop_group in "
    "['treated_high_cornell_route_stop', 'control_low_cornell_route_stop']"
).copy()
didisc_route_stop_panel["treated"] = didisc_route_stop_panel["route_stop_group"].eq(
    "treated_high_cornell_route_stop"
).astype(int)
didisc_route_stop_panel["post"] = didisc_route_stop_panel["post_6pm"].astype(int)
didisc_route_stop_panel["running_minutes"] = didisc_route_stop_panel["minutes_since_18"]

didisc_route_stop_panel.head()

In [ ]:
# Compact row-level analysis-ready ride-events dataframe.
ride_events = pd.read_parquet(ANALYSIS_READY_DIR / "ride_events_analysis_ready.parquet")

display(ride_events.head())
ride_events[[
    "event_time",
    "route_id",
    "route_name",
    "stop_id",
    "stop_name",
    "fare_category",
]].isna().sum()

The `ride_events` dataframe keeps one cleaned boarding/event row per retained record. It drops unresolved routes, missing/placeholder stops, `Empty TTP`, upstream-excluded text records, and duplicate fare events. It keeps blank APC-only records as `Unspecified / No Description` because they are valid boardings with missing fare detail.


### Cleaning choices to carry forward

Use `route_panel` for route-level models and `route_stop_panel` for the preferred route-stop exposure design. The main outcome should be `boards`. Use `non_cornell_card_boards` as a sensitivity outcome if you want to remove already-unlimited Cornell Card riders.

For fare-media checks, use `cornell_override_boards` and `off_policy_cornell_override_boards`; do not use those fields to define treatment. The panels include issue columns (`dropfile_exception_boards`, `fare_record_missing_description_boards`, `apc_only_missing_fare_description_boards`) plus family columns (`free_boards`, `half_fare_boards`, `regular_fares_boards`) and detailed category columns (`free_fares_boards`, `transfer_boards`, `ic_boards`, `ride_based_half_fares_boards`, `ride_based_regular_fares_boards`, `period_based_regular_fares_boards`, etc.).


## Direction Column Diagnostics

The `direction_vhist` column is not a stop ID. It behaves like a vehicle compass heading: integer values from 0 to 360, mostly every 2 degrees. The stop IDs are in `Stop_Id_vhist` and `Previous_Stop_Id_vhist`.

`Inbound_Outbound_vhist` is also not a stop ID. It is a small categorical vehicle-history direction code, with values like 3, 4, 6, 7, and 8 depending on route.


In [ ]:
DIRECTION_DIAG_DIR = PROCESSED_DIR / "_Archive" / "Diagnostics" / "Direction_Diagnostics"

# Rebuild with:
# /Users/stevenzhou/miniconda3/bin/conda run -n spatial python scripts/_archive_diagnostics/investigate_direction_column.py

direction_summary = pd.read_csv(
    DIRECTION_DIAG_DIR / "direction_column_summary.csv", index_col=0
).rename(columns={"value": "rows_or_value"})
direction_counts = pd.read_csv(DIRECTION_DIAG_DIR / "direction_value_counts.csv")
inbound_outbound_by_route = pd.read_csv(DIRECTION_DIAG_DIR / "inbound_outbound_by_route.csv")
direction_sample = pd.read_csv(DIRECTION_DIAG_DIR / "direction_rows_sample.csv")

display(direction_summary)
display(direction_counts.head(20))
display(inbound_outbound_by_route.head(40))
direction_sample.head(20)

Interpretation:

- Use `Stop_Id_vhist` for stops.
- Use `direction_vhist` only as a compass/bearing measure, not as a route direction or stop identifier.
- If a model needs route-direction controls, `Inbound_Outbound_vhist` is the better candidate, but treat it as a route-specific categorical code rather than a globally meaningful inbound/outbound label.
